In [199]:
import sys
sys.path.append("../../")
from src.models.bart_mnli import BartForSequenceClassification, BartMnliConfig
import torch
import pandas as pd
import json
import ast
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from experiments.multiturn.parse import (
    SEG_DONE,
    SEG_IGNORE,
    parse_conversations,
    parse_tools,
    segment_assistant,
    extract_tool_calls
)
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

In [2]:
model = BartForSequenceClassification(BartMnliConfig())

In [3]:
model.load_state_dict(torch.load(r"C:\Users\tsuma.thomas\Documents\CoreOutline\transformer\models\bart_mnli_tool_selector.pth"), strict=True)

<All keys matched successfully>

In [4]:
model.eval()

BartForSequenceClassification(
  (model): BartModel(
    (shared): Embedding(50265, 1024, padding_idx=1)
    (encoder): BartEncoder(
      (embed_tokens): Embedding(50265, 1024, padding_idx=1)
      (embed_positions): BartLearnedPositionalEmbedding(1026, 1024)
      (layernorm_embedding): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      (layers): ModuleList(
        (0-11): 12 x BartEncoderLayer(
          (self_attn): BartAttention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear(in_features=4096, out_features=1024, bias=T

In [6]:
HF_NAME = "facebook/bart-large-mnli"

tok = AutoTokenizer.from_pretrained(HF_NAME)
premise = "What is the weather in Tokyo tomorrow?"
hypothesis = ["This request requires a tool that can get weather forecasts.", "This request requires a tool to create a google doc"]
enc = [ tok(premise, i, return_tensors="pt", truncation=True) for i in hypothesis ]

In [7]:
recon_logits = [ model(input_ids=i["input_ids"], attention_mask=i["attention_mask"]) for i in enc ]

In [8]:
recon_logits

[tensor([[-3.3940,  3.3342, -0.1094]], grad_fn=<AddmmBackward0>),
 tensor([[-1.7785,  3.8149, -1.9174]], grad_fn=<AddmmBackward0>)]

In [9]:
id2label = {0: "contradiction", 1: "neutral", 2: "entailment"}

In [10]:
recon_pred = [ id2label[int(i.argmax())] for i in recon_logits ]

In [11]:
recon_pred

['neutral', 'neutral']

In [12]:
data = pd.read_csv(r"C:\Users\tsuma.thomas\Documents\CoreOutline\transformer\data\tool-use-multiturn-reasoning.csv")

In [13]:
data

,Unnamed: 0,conversations,tools,task,category,source
0,0,"[{'from': 'system', 'value': 'You are a deep t...","[{""name"": ""get_future_events"", ""description"": ...",Could you provide me with a list of upcoming A...,Get Future Events,ToolAce
1,1,"[{'from': 'system', 'value': 'You are a deep t...","[{""name"": ""get_future_events"", ""description"": ...",Could you provide me with a list of upcoming A...,Get Future Events,ToolAce
2,2,"[{'from': 'system', 'value': 'You are a deep t...","[{""name"": ""get_future_events"", ""description"": ...",Could you provide me with a list of upcoming A...,Get Future Events,ToolAce
3,3,"[{'from': 'system', 'value': 'You are a deep t...","[{""name"": ""get_future_events"", ""description"": ...",Could you provide me with a list of upcoming A...,Get Future Events,ToolAce
4,4,"[{'from': 'system', 'value': 'You are a deep t...","[{""name"": ""get_future_events"", ""description"": ...",Could you provide me with a list of upcoming A...,Get Future Events,ToolAce
...,...,...,...,...,...,...
14574,14574,"[{'from': 'system', 'value': ""You are a deep t...","[{""type"": ""function"", ""function"": {""name"": ""ge...","Can you tell me about the movie ""Inception""?",Get Movie Details,Glaive
14575,14575,"[{'from': 'system', 'value': ""You are a deep t...","[{""type"": ""function"", ""function"": {""name"": ""ge...","Can you tell me about the movie ""Inception""?",Get Movie Details,Glaive
14576,14576,"[{'from': 'system', 'value': ""You are a deep t...","[{""type"": ""function"", ""function"": {""name"": ""se...",I'm in New York and I'm craving for some Itali...,Search Restaurants,Glaive
14577,14577,"[{'from': 'system', 'value': ""You are a deep t...","[{""type"": ""function"", ""function"": {""name"": ""se...",I'm in New York and I'm craving for some Itali...,Search Restaurants,Glaive


In [14]:
conversation = parse_conversations(data['conversations'][0])

In [15]:
"\n".join([ i['value'] for i in conversation[:2] ])

'You are a deep thinking AI, you may use extremely long chains of thought to deeply consider the problem and deliberate with yourself via systematic reasoning processes to help come to a correct solution prior to answering. You should enclose your thoughts and internal monologue inside <think> </think> tags, and then provide your solution or response to the problem.\n\nYou are a function calling AI model. You are provided with function signatures within <tools> </tools> XML tags. You may call one or more functions to assist with the user query. If available tools are not relevant in assisting with user query, just respond in natural conversational language. Don\'t make assumptions about what values to plug into functions. After calling & executing the functions, you will be provided with function results within <tool_response> </tool_response> XML tags. Here are the available tools:\n<tools>\n[{"name":"get_future_events","description":"Retrieve a list of future Azure events, such as ma

In [38]:
import re
import json

def get_tools(x):
    input_string = "\n".join([ i['value'] for i in x ])
    
    # Use regex to find the content inside <tools> tags
    match = re.search(r'<tools>\n(.*)\n</tools>', input_string, re.DOTALL)
    
    if match:
        tools_json_str = match.group(1)
        # Parse the JSON string into a Python list
        tools_list = json.loads(tools_json_str)
        return tools_list
        
        # # Display the tools
        # for tool in tools_list:
        #     print("\n\n")
        #     print(f"Name: {tool['name']}")
        #     print(f"Description: {tool['description']}\n")
        #     # print(tool)
    else:
        print("No tools found.")
tools_list = get_tools(conversation[:2])

In [39]:
tools_list

[{'name': 'get_future_events',
  'description': 'Retrieve a list of future Azure events, such as maintenance windows, upstrings, or other scheduled events.',
  'parameters': {'type': 'dict',
   'properties': {'page': {'description': 'The page number to retrieve (default: 1)',
     'type': 'int'}},
   'required': ['page']},
  'required': None},
 {'name': 'get_languages_for_country',
  'description': 'Get a list of valid languages for a given country code.',
  'parameters': {'type': 'dict',
   'properties': {'country': {'description': 'Country code of the country to get languages for. See [all available country codes](https://en.wikipedia.org/wiki/ISO_3166-1_alpha-2).',
     'type': 'string',
     'default': 'US'}},
   'required': ['country']},
  'required': None},
 {'name': 'get_all_dog_breeds',
  'description': 'This endpoint returns a list of all available dog breeds, along with their relevant information.',
  'parameters': {'type': 'dict', 'properties': {}, 'required': []},
  'requir

In [17]:
json.loads(data['tools'][0])

[{'name': 'get_future_events',
  'description': 'Retrieve a list of future Azure events, such as maintenance windows, upstrings, or other scheduled events.',
  'parameters': {'type': 'dict',
   'properties': {'page': {'description': 'The page number to retrieve (default: 1)',
     'type': 'int'}},
   'required': ['page']},
  'required': None},
 {'name': 'get_languages_for_country',
  'description': 'Get a list of valid languages for a given country code.',
  'parameters': {'type': 'dict',
   'properties': {'country': {'description': 'Country code of the country to get languages for. See [all available country codes](https://en.wikipedia.org/wiki/ISO_3166-1_alpha-2).',
     'type': 'string',
     'default': 'US'}},
   'required': ['country']},
  'required': None},
 {'name': 'get_all_dog_breeds',
  'description': 'This endpoint returns a list of all available dog breeds, along with their relevant information.',
  'parameters': {'type': 'dict', 'properties': {}, 'required': []},
  'requir

In [18]:
len(tools)

NameError: name 'tools' is not defined

In [19]:
tools[0]

NameError: name 'tools' is not defined

In [20]:
tools_list

[{'name': 'get_future_events',
  'description': 'Retrieve a list of future Azure events, such as maintenance windows, upstrings, or other scheduled events.',
  'parameters': {'type': 'dict',
   'properties': {'page': {'description': 'The page number to retrieve (default: 1)',
     'type': 'int'}},
   'required': ['page']},
  'required': None},
 {'name': 'get_languages_for_country',
  'description': 'Get a list of valid languages for a given country code.',
  'parameters': {'type': 'dict',
   'properties': {'country': {'description': 'Country code of the country to get languages for. See [all available country codes](https://en.wikipedia.org/wiki/ISO_3166-1_alpha-2).',
     'type': 'string',
     'default': 'US'}},
   'required': ['country']},
  'required': None},
 {'name': 'get_all_dog_breeds',
  'description': 'This endpoint returns a list of all available dog breeds, along with their relevant information.',
  'parameters': {'type': 'dict', 'properties': {}, 'required': []},
  'requir

In [21]:
descriptions = [ i['description'] for i in tools_list ]

In [22]:
request = conversation[1]['value']

In [23]:
enc = [ tok(request, i, return_tensors="pt", truncation=True) for i in descriptions ]

In [163]:
enc

[{'input_ids': tensor([[    0, 35299,    47,   694,   162,    19,    10,   889,     9,  2568,
          25959,  1061,   116,  3401,   386,    19,     5,    78,  1842,     9,
            775,     4,     2,     2, 27814, 20080,    10,   889,     9,   499,
          25959,  1061,     6,   215,    25,  4861,  6410,     6,    62, 40563,
              6,    50,    97,  1768,  1061,     4,     2]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
          1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])},
 {'input_ids': tensor([[    0, 35299,    47,   694,   162,    19,    10,   889,     9,  2568,
          25959,  1061,   116,  3401,   386,    19,     5,    78,  1842,     9,
            775,     4,     2,     2, 14181,    10,   889,     9,  8218, 11991,
             13,    10,   576,   247,  3260,     4,     2]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       

In [24]:
_logits = [ model(input_ids=i["input_ids"], attention_mask=i["attention_mask"]) for i in enc ]

In [25]:
_logits

[tensor([[-2.8419,  5.0818, -1.6050]], grad_fn=<AddmmBackward0>),
 tensor([[ 2.5219, -0.4366, -2.6250]], grad_fn=<AddmmBackward0>),
 tensor([[ 3.0662,  0.7896, -3.2485]], grad_fn=<AddmmBackward0>),
 tensor([[ 2.0708,  0.0483, -1.2345]], grad_fn=<AddmmBackward0>),
 tensor([[ 2.9032, -0.8210, -2.6478]], grad_fn=<AddmmBackward0>),
 tensor([[ 0.0782,  1.1788, -0.9813]], grad_fn=<AddmmBackward0>)]

In [26]:
_pred = [ id2label[int(i.argmax())] for i in _logits ]

In [27]:
_pred

['neutral',
 'contradiction',
 'contradiction',
 'contradiction',
 'contradiction',
 'neutral']

In [28]:
max([ (i.max()) for i in _logits ])

tensor(5.0818, grad_fn=<MaxBackward1>)

In [29]:
data

,Unnamed: 0,conversations,tools,task,category,source
0,0,"[{'from': 'system', 'value': 'You are a deep t...","[{""name"": ""get_future_events"", ""description"": ...",Could you provide me with a list of upcoming A...,Get Future Events,ToolAce
1,1,"[{'from': 'system', 'value': 'You are a deep t...","[{""name"": ""get_future_events"", ""description"": ...",Could you provide me with a list of upcoming A...,Get Future Events,ToolAce
2,2,"[{'from': 'system', 'value': 'You are a deep t...","[{""name"": ""get_future_events"", ""description"": ...",Could you provide me with a list of upcoming A...,Get Future Events,ToolAce
3,3,"[{'from': 'system', 'value': 'You are a deep t...","[{""name"": ""get_future_events"", ""description"": ...",Could you provide me with a list of upcoming A...,Get Future Events,ToolAce
4,4,"[{'from': 'system', 'value': 'You are a deep t...","[{""name"": ""get_future_events"", ""description"": ...",Could you provide me with a list of upcoming A...,Get Future Events,ToolAce
...,...,...,...,...,...,...
14574,14574,"[{'from': 'system', 'value': ""You are a deep t...","[{""type"": ""function"", ""function"": {""name"": ""ge...","Can you tell me about the movie ""Inception""?",Get Movie Details,Glaive
14575,14575,"[{'from': 'system', 'value': ""You are a deep t...","[{""type"": ""function"", ""function"": {""name"": ""ge...","Can you tell me about the movie ""Inception""?",Get Movie Details,Glaive
14576,14576,"[{'from': 'system', 'value': ""You are a deep t...","[{""type"": ""function"", ""function"": {""name"": ""se...",I'm in New York and I'm craving for some Itali...,Search Restaurants,Glaive
14577,14577,"[{'from': 'system', 'value': ""You are a deep t...","[{""type"": ""function"", ""function"": {""name"": ""se...",I'm in New York and I'm craving for some Itali...,Search Restaurants,Glaive


In [60]:
data['conversations_json'] = data['conversations'].apply(parse_conversations)

In [61]:
data['conversations_json'][0]

[{'from': 'system',
  'value': 'You are a deep thinking AI, you may use extremely long chains of thought to deeply consider the problem and deliberate with yourself via systematic reasoning processes to help come to a correct solution prior to answering. You should enclose your thoughts and internal monologue inside <think> </think> tags, and then provide your solution or response to the problem.\n\nYou are a function calling AI model. You are provided with function signatures within <tools> </tools> XML tags. You may call one or more functions to assist with the user query. If available tools are not relevant in assisting with user query, just respond in natural conversational language. Don\'t make assumptions about what values to plug into functions. After calling & executing the functions, you will be provided with function results within <tool_response> </tool_response> XML tags. Here are the available tools:\n<tools>\n[{"name":"get_future_events","description":"Retrieve a list of 

In [51]:
# data['conversations_json'] = [ i[0]['value'] for i in data['conversations_json'] ]

In [83]:
import re
import json

def get_tools(x):
    input_string = x[0]['value']
    # Use regex to find the content inside <tools> tags
    match = re.search(r'<tools>\n(.*)\n</tools>', input_string, re.DOTALL)

    # try:
    if match:
        tools_json_str = match.group(1)
        # Parse the JSON string into a Python list
        tools_list = json.loads(tools_json_str)
        return tools_list
        
        # # Display the tools
        # for tool in tools_list:
        #     print("\n\n")
        #     print(f"Name: {tool['name']}")
        #     print(f"Description: {tool['description']}\n")
        #     # print(tool)
    else:
        print("No tools found.")
    # except:
    #     return None

In [84]:
get_tools

<function __main__.get_tools(x)>

In [85]:
data['conversations_tools'] = data['conversations_json'].apply(get_tools)

No tools found.
No tools found.
No tools found.
No tools found.
No tools found.
No tools found.
No tools found.
No tools found.
No tools found.
No tools found.
No tools found.
No tools found.
No tools found.
No tools found.
No tools found.
No tools found.
No tools found.
No tools found.
No tools found.
No tools found.
No tools found.
No tools found.
No tools found.
No tools found.
No tools found.
No tools found.
No tools found.
No tools found.
No tools found.
No tools found.
No tools found.
No tools found.
No tools found.
No tools found.
No tools found.
No tools found.
No tools found.
No tools found.
No tools found.
No tools found.
No tools found.
No tools found.
No tools found.
No tools found.
No tools found.
No tools found.
No tools found.
No tools found.
No tools found.
No tools found.
No tools found.
No tools found.
No tools found.
No tools found.
No tools found.
No tools found.
No tools found.
No tools found.
No tools found.
No tools found.
No tools found.
No tools found.
No tools

In [94]:
data = pd.read_csv(r"C:\Users\tsuma.thomas\Documents\CoreOutline\transformer\data\tool-use-multiturn-reasoning-with-tool-list.csv")

In [95]:
data['tool_list']

0        [{"type": "function", "function": {"name": "ge...
1        [{"type": "function", "function": {"name": "ge...
2        [{"type": "function", "function": {"name": "ge...
3        [{"type": "function", "function": {"name": "ge...
4        [{"type": "function", "function": {"name": "ge...
                               ...                        
14574    [{"type": "function", "function": {"name": "ge...
14575    [{"type": "function", "function": {"name": "ge...
14576    [{"type": "function", "function": {"name": "se...
14577    [{"type": "function", "function": {"name": "se...
14578    [{"type": "function", "function": {"name": "se...
Name: tool_list, Length: 14579, dtype: object

In [99]:
[ i['function']['description'] for i in json.loads(data['tool_list'][0]) ]

['Retrieve a list of future Azure events, such as maintenance windows, upstrings, or other scheduled events.',
 'Get a list of valid languages for a given country code.',
 'This endpoint returns a list of all available dog breeds, along with their relevant information.',
 "Retrieves a LinkedIn user's prostring data, including experience, education history, skills, and company-related details, given their Sales Navigator URL.",
 'Retrieve a list of the top 100 companies that are related to a given SIC code.',
 "Returns every verse containing the supplied Strong's number. Include LXX boolean option allows searching the Septuagint translation of the Old Testament when searching for a Greek word, enabling connections between New Testament words and Old Testament concepts."]

In [102]:
json.loads(data['tools'][0])

[{'name': 'get_future_events',
  'description': 'Retrieve a list of future Azure events, such as maintenance windows, upstrings, or other scheduled events.',
  'parameters': {'type': 'dict',
   'properties': {'page': {'description': 'The page number to retrieve (default: 1)',
     'type': 'int'}},
   'required': ['page']},
  'required': None},
 {'name': 'get_languages_for_country',
  'description': 'Get a list of valid languages for a given country code.',
  'parameters': {'type': 'dict',
   'properties': {'country': {'description': 'Country code of the country to get languages for. See [all available country codes](https://en.wikipedia.org/wiki/ISO_3166-1_alpha-2).',
     'type': 'string',
     'default': 'US'}},
   'required': ['country']},
  'required': None},
 {'name': 'get_all_dog_breeds',
  'description': 'This endpoint returns a list of all available dog breeds, along with their relevant information.',
  'parameters': {'type': 'dict', 'properties': {}, 'required': []},
  'requir

In [107]:
conversation

[{'from': 'system',
  'value': 'You are a deep thinking AI, you may use extremely long chains of thought to deeply consider the problem and deliberate with yourself via systematic reasoning processes to help come to a correct solution prior to answering. You should enclose your thoughts and internal monologue inside <think> </think> tags, and then provide your solution or response to the problem.\n\nYou are a function calling AI model. You are provided with function signatures within <tools> </tools> XML tags. You may call one or more functions to assist with the user query. If available tools are not relevant in assisting with user query, just respond in natural conversational language. Don\'t make assumptions about what values to plug into functions. After calling & executing the functions, you will be provided with function results within <tool_response> </tool_response> XML tags. Here are the available tools:\n<tools>\n[{"name":"get_future_events","description":"Retrieve a list of 

In [108]:
[ i for i in conversation if i['from'] != 'gpt' ]

[{'from': 'system',
  'value': 'You are a deep thinking AI, you may use extremely long chains of thought to deeply consider the problem and deliberate with yourself via systematic reasoning processes to help come to a correct solution prior to answering. You should enclose your thoughts and internal monologue inside <think> </think> tags, and then provide your solution or response to the problem.\n\nYou are a function calling AI model. You are provided with function signatures within <tools> </tools> XML tags. You may call one or more functions to assist with the user query. If available tools are not relevant in assisting with user query, just respond in natural conversational language. Don\'t make assumptions about what values to plug into functions. After calling & executing the functions, you will be provided with function results within <tool_response> </tool_response> XML tags. Here are the available tools:\n<tools>\n[{"name":"get_future_events","description":"Retrieve a list of 

In [110]:
data['task']

0        Could you provide me with a list of upcoming A...
1        Could you provide me with a list of upcoming A...
2        Could you provide me with a list of upcoming A...
3        Could you provide me with a list of upcoming A...
4        Could you provide me with a list of upcoming A...
                               ...                        
14574         Can you tell me about the movie "Inception"?
14575         Can you tell me about the movie "Inception"?
14576    I'm in New York and I'm craving for some Itali...
14577    I'm in New York and I'm craving for some Itali...
14578    I'm in New York and I'm craving for some Itali...
Name: task, Length: 14579, dtype: object

In [112]:
json.loads(data['tools'][0])

[{'name': 'get_future_events',
  'description': 'Retrieve a list of future Azure events, such as maintenance windows, upstrings, or other scheduled events.',
  'parameters': {'type': 'dict',
   'properties': {'page': {'description': 'The page number to retrieve (default: 1)',
     'type': 'int'}},
   'required': ['page']},
  'required': None},
 {'name': 'get_languages_for_country',
  'description': 'Get a list of valid languages for a given country code.',
  'parameters': {'type': 'dict',
   'properties': {'country': {'description': 'Country code of the country to get languages for. See [all available country codes](https://en.wikipedia.org/wiki/ISO_3166-1_alpha-2).',
     'type': 'string',
     'default': 'US'}},
   'required': ['country']},
  'required': None},
 {'name': 'get_all_dog_breeds',
  'description': 'This endpoint returns a list of all available dog breeds, along with their relevant information.',
  'parameters': {'type': 'dict', 'properties': {}, 'required': []},
  'requir

In [113]:
json.loads(data['tool_list'][0])

[{'type': 'function',
  'function': {'name': 'get_future_events',
   'description': 'Retrieve a list of future Azure events, such as maintenance windows, upstrings, or other scheduled events.',
   'parameters': {'type': 'object',
    'properties': {'page': {'description': 'The page number to retrieve (default: 1)',
      'type': 'integer'}},
    'required': ['page']}}},
 {'type': 'function',
  'function': {'name': 'get_languages_for_country',
   'description': 'Get a list of valid languages for a given country code.',
   'parameters': {'type': 'object',
    'properties': {'country': {'description': 'Country code of the country to get languages for. See [all available country codes](https://en.wikipedia.org/wiki/ISO_3166-1_alpha-2).',
      'type': 'string',
      'default': 'US'}},
    'required': ['country']}}},
 {'type': 'function',
  'function': {'name': 'get_all_dog_breeds',
   'description': 'This endpoint returns a list of all available dog breeds, along with their relevant infor

In [127]:
data = pd.read_csv(r"C:\Users\tsuma.thomas\Documents\CoreOutline\transformer\data\tool-use-multiturn-expanded.csv")

In [128]:
data

,Unnamed: 0,conversations,tools,task,category,source,tool_list,conversation_so_far,selected_tools
0,0,"[{'from': 'system', 'value': 'You are a deep t...","[{""name"": ""get_future_events"", ""description"": ...",Could you provide me with a list of upcoming A...,Get Future Events,ToolAce,"[{""type"": ""function"", ""function"": {""name"": ""ge...","[{""from"": ""system"", ""value"": ""You are a deep t...","[""get_future_events""]"
1,0,"[{'from': 'system', 'value': 'You are a deep t...","[{""name"": ""get_future_events"", ""description"": ...",Could you provide me with a list of upcoming A...,Get Future Events,ToolAce,"[{""type"": ""function"", ""function"": {""name"": ""ge...","[{""from"": ""system"", ""value"": ""You are a deep t...","[""get_languages_for_country""]"
2,1,"[{'from': 'system', 'value': 'You are a deep t...","[{""name"": ""get_future_events"", ""description"": ...",Could you provide me with a list of upcoming A...,Get Future Events,ToolAce,"[{""type"": ""function"", ""function"": {""name"": ""ge...","[{""from"": ""system"", ""value"": ""You are a deep t...","[""get_future_events""]"
3,1,"[{'from': 'system', 'value': 'You are a deep t...","[{""name"": ""get_future_events"", ""description"": ...",Could you provide me with a list of upcoming A...,Get Future Events,ToolAce,"[{""type"": ""function"", ""function"": {""name"": ""ge...","[{""from"": ""system"", ""value"": ""You are a deep t...","[""get_languages_for_country""]"
4,2,"[{'from': 'system', 'value': 'You are a deep t...","[{""name"": ""get_future_events"", ""description"": ...",Could you provide me with a list of upcoming A...,Get Future Events,ToolAce,"[{""type"": ""function"", ""function"": {""name"": ""ge...","[{""from"": ""system"", ""value"": ""You are a deep t...","[""get_future_events""]"
...,...,...,...,...,...,...,...,...,...
32660,14576,"[{'from': 'system', 'value': ""You are a deep t...","[{""type"": ""function"", ""function"": {""name"": ""se...",I'm in New York and I'm craving for some Itali...,Search Restaurants,Glaive,"[{""type"": ""function"", ""function"": {""name"": ""se...","[{""from"": ""system"", ""value"": ""You are a deep t...","[""search_restaurants""]"
32661,14577,"[{'from': 'system', 'value': ""You are a deep t...","[{""type"": ""function"", ""function"": {""name"": ""se...",I'm in New York and I'm craving for some Itali...,Search Restaurants,Glaive,"[{""type"": ""function"", ""function"": {""name"": ""se...","[{""from"": ""system"", ""value"": ""You are a deep t...","[""search_restaurants""]"
32662,14577,"[{'from': 'system', 'value': ""You are a deep t...","[{""type"": ""function"", ""function"": {""name"": ""se...",I'm in New York and I'm craving for some Itali...,Search Restaurants,Glaive,"[{""type"": ""function"", ""function"": {""name"": ""se...","[{""from"": ""system"", ""value"": ""You are a deep t...","[""search_restaurants""]"
32663,14578,"[{'from': 'system', 'value': ""You are a deep t...","[{""type"": ""function"", ""function"": {""name"": ""se...",I'm in New York and I'm craving for some Itali...,Search Restaurants,Glaive,"[{""type"": ""function"", ""function"": {""name"": ""se...","[{""from"": ""system"", ""value"": ""You are a deep t...","[""search_restaurants""]"


In [130]:
json.loads(data['conversation_so_far'][0])

[{'from': 'system',
  'value': "You are a deep thinking AI, you may use extremely long chains of thought to deeply consider the problem and deliberate with yourself via systematic reasoning processes to help come to a correct solution prior to answering. You should enclose your thoughts and internal monologue inside <think> </think> tags, and then provide your solution or response to the problem.\n\nYou are a function calling AI model. You are provided with function signatures within <tools> </tools> XML tags. You may call one or more functions to assist with the user query. If available tools are not relevant in assisting with user query, just respond in natural conversational language. Don't make assumptions about what values to plug into functions. After calling & executing the functions, you will be provided with function results within <tool_response> </tool_response> XML tags. For each function call return a JSON object, with the following pydantic model json schema for each:\n{'

In [140]:
[ i for i in json.loads(data['conversation_so_far'][3])]

[{'from': 'system',
  'value': "You are a deep thinking AI, you may use extremely long chains of thought to deeply consider the problem and deliberate with yourself via systematic reasoning processes to help come to a correct solution prior to answering. You should enclose your thoughts and internal monologue inside <think> </think> tags, and then provide your solution or response to the problem.\n\nYou are a function calling AI model. You are provided with function signatures within <tools> </tools> XML tags. You may call one or more functions to assist with the user query. If available tools are not relevant in assisting with user query, just respond in natural conversational language. Don't make assumptions about what values to plug into functions. After calling & executing the functions, you will be provided with function results within <tool_response> </tool_response> XML tags. For each function call return a JSON object, with the following pydantic model json schema for each:\n{'

In [147]:
data['conversation_so_far_str'] = [  " ".join([ j['value'] for j in json.loads(i)]) for i in data['conversation_so_far'] ]

In [148]:
data

,Unnamed: 0,conversations,tools,task,category,source,tool_list,conversation_so_far,selected_tools,conversation_so_far_str
0,0,"[{'from': 'system', 'value': 'You are a deep t...","[{""name"": ""get_future_events"", ""description"": ...",Could you provide me with a list of upcoming A...,Get Future Events,ToolAce,"[{""type"": ""function"", ""function"": {""name"": ""ge...","[{""from"": ""system"", ""value"": ""You are a deep t...","[""get_future_events""]","You are a deep thinking AI, you may use extrem..."
1,0,"[{'from': 'system', 'value': 'You are a deep t...","[{""name"": ""get_future_events"", ""description"": ...",Could you provide me with a list of upcoming A...,Get Future Events,ToolAce,"[{""type"": ""function"", ""function"": {""name"": ""ge...","[{""from"": ""system"", ""value"": ""You are a deep t...","[""get_languages_for_country""]","You are a deep thinking AI, you may use extrem..."
2,1,"[{'from': 'system', 'value': 'You are a deep t...","[{""name"": ""get_future_events"", ""description"": ...",Could you provide me with a list of upcoming A...,Get Future Events,ToolAce,"[{""type"": ""function"", ""function"": {""name"": ""ge...","[{""from"": ""system"", ""value"": ""You are a deep t...","[""get_future_events""]","You are a deep thinking AI, you may use extrem..."
3,1,"[{'from': 'system', 'value': 'You are a deep t...","[{""name"": ""get_future_events"", ""description"": ...",Could you provide me with a list of upcoming A...,Get Future Events,ToolAce,"[{""type"": ""function"", ""function"": {""name"": ""ge...","[{""from"": ""system"", ""value"": ""You are a deep t...","[""get_languages_for_country""]","You are a deep thinking AI, you may use extrem..."
4,2,"[{'from': 'system', 'value': 'You are a deep t...","[{""name"": ""get_future_events"", ""description"": ...",Could you provide me with a list of upcoming A...,Get Future Events,ToolAce,"[{""type"": ""function"", ""function"": {""name"": ""ge...","[{""from"": ""system"", ""value"": ""You are a deep t...","[""get_future_events""]","You are a deep thinking AI, you may use extrem..."
...,...,...,...,...,...,...,...,...,...,...
32660,14576,"[{'from': 'system', 'value': ""You are a deep t...","[{""type"": ""function"", ""function"": {""name"": ""se...",I'm in New York and I'm craving for some Itali...,Search Restaurants,Glaive,"[{""type"": ""function"", ""function"": {""name"": ""se...","[{""from"": ""system"", ""value"": ""You are a deep t...","[""search_restaurants""]","You are a deep thinking AI, you may use extrem..."
32661,14577,"[{'from': 'system', 'value': ""You are a deep t...","[{""type"": ""function"", ""function"": {""name"": ""se...",I'm in New York and I'm craving for some Itali...,Search Restaurants,Glaive,"[{""type"": ""function"", ""function"": {""name"": ""se...","[{""from"": ""system"", ""value"": ""You are a deep t...","[""search_restaurants""]","You are a deep thinking AI, you may use extrem..."
32662,14577,"[{'from': 'system', 'value': ""You are a deep t...","[{""type"": ""function"", ""function"": {""name"": ""se...",I'm in New York and I'm craving for some Itali...,Search Restaurants,Glaive,"[{""type"": ""function"", ""function"": {""name"": ""se...","[{""from"": ""system"", ""value"": ""You are a deep t...","[""search_restaurants""]","You are a deep thinking AI, you may use extrem..."
32663,14578,"[{'from': 'system', 'value': ""You are a deep t...","[{""type"": ""function"", ""function"": {""name"": ""se...",I'm in New York and I'm craving for some Itali...,Search Restaurants,Glaive,"[{""type"": ""function"", ""function"": {""name"": ""se...","[{""from"": ""system"", ""value"": ""You are a deep t...","[""search_restaurants""]","You are a deep thinking AI, you may use extrem..."


In [156]:
data['tool_descriptions'] = [ [ j['function']['description'] for j in json.loads(i) ] for i in data['tool_list'] ]

In [157]:
data

,Unnamed: 0,conversations,tools,task,category,source,tool_list,conversation_so_far,selected_tools,conversation_so_far_str,tool_descriptions
0,0,"[{'from': 'system', 'value': 'You are a deep t...","[{""name"": ""get_future_events"", ""description"": ...",Could you provide me with a list of upcoming A...,Get Future Events,ToolAce,"[{""type"": ""function"", ""function"": {""name"": ""ge...","[{""from"": ""system"", ""value"": ""You are a deep t...","[""get_future_events""]","You are a deep thinking AI, you may use extrem...","[Retrieve a list of future Azure events, such ..."
1,0,"[{'from': 'system', 'value': 'You are a deep t...","[{""name"": ""get_future_events"", ""description"": ...",Could you provide me with a list of upcoming A...,Get Future Events,ToolAce,"[{""type"": ""function"", ""function"": {""name"": ""ge...","[{""from"": ""system"", ""value"": ""You are a deep t...","[""get_languages_for_country""]","You are a deep thinking AI, you may use extrem...","[Retrieve a list of future Azure events, such ..."
2,1,"[{'from': 'system', 'value': 'You are a deep t...","[{""name"": ""get_future_events"", ""description"": ...",Could you provide me with a list of upcoming A...,Get Future Events,ToolAce,"[{""type"": ""function"", ""function"": {""name"": ""ge...","[{""from"": ""system"", ""value"": ""You are a deep t...","[""get_future_events""]","You are a deep thinking AI, you may use extrem...","[Retrieve a list of future Azure events, such ..."
3,1,"[{'from': 'system', 'value': 'You are a deep t...","[{""name"": ""get_future_events"", ""description"": ...",Could you provide me with a list of upcoming A...,Get Future Events,ToolAce,"[{""type"": ""function"", ""function"": {""name"": ""ge...","[{""from"": ""system"", ""value"": ""You are a deep t...","[""get_languages_for_country""]","You are a deep thinking AI, you may use extrem...","[Retrieve a list of future Azure events, such ..."
4,2,"[{'from': 'system', 'value': 'You are a deep t...","[{""name"": ""get_future_events"", ""description"": ...",Could you provide me with a list of upcoming A...,Get Future Events,ToolAce,"[{""type"": ""function"", ""function"": {""name"": ""ge...","[{""from"": ""system"", ""value"": ""You are a deep t...","[""get_future_events""]","You are a deep thinking AI, you may use extrem...","[Retrieve a list of future Azure events, such ..."
...,...,...,...,...,...,...,...,...,...,...,...
32660,14576,"[{'from': 'system', 'value': ""You are a deep t...","[{""type"": ""function"", ""function"": {""name"": ""se...",I'm in New York and I'm craving for some Itali...,Search Restaurants,Glaive,"[{""type"": ""function"", ""function"": {""name"": ""se...","[{""from"": ""system"", ""value"": ""You are a deep t...","[""search_restaurants""]","You are a deep thinking AI, you may use extrem...",[Search for restaurants based on location and ...
32661,14577,"[{'from': 'system', 'value': ""You are a deep t...","[{""type"": ""function"", ""function"": {""name"": ""se...",I'm in New York and I'm craving for some Itali...,Search Restaurants,Glaive,"[{""type"": ""function"", ""function"": {""name"": ""se...","[{""from"": ""system"", ""value"": ""You are a deep t...","[""search_restaurants""]","You are a deep thinking AI, you may use extrem...",[Search for restaurants based on location and ...
32662,14577,"[{'from': 'system', 'value': ""You are a deep t...","[{""type"": ""function"", ""function"": {""name"": ""se...",I'm in New York and I'm craving for some Itali...,Search Restaurants,Glaive,"[{""type"": ""function"", ""function"": {""name"": ""se...","[{""from"": ""system"", ""value"": ""You are a deep t...","[""search_restaurants""]","You are a deep thinking AI, you may use extrem...",[Search for restaurants based on location and ...
32663,14578,"[{'from': 'system', 'value': ""You are a deep t...","[{""type"": ""function"", ""function"": {""name"": ""se...",I'm in New York and I'm craving for some Itali...,Search Restaurants,Glaive,"[{""type"": ""function"", ""function"": 

In [158]:
data['tool_names'] = [ [ j['function']['name'] for j in json.loads(i) ] for i in data['tool_list'] ]

In [159]:
data

,Unnamed: 0,conversations,tools,task,category,source,tool_list,conversation_so_far,selected_tools,conversation_so_far_str,tool_descriptions,tool_names
0,0,"[{'from': 'system', 'value': 'You are a deep t...","[{""name"": ""get_future_events"", ""description"": ...",Could you provide me with a list of upcoming A...,Get Future Events,ToolAce,"[{""type"": ""function"", ""function"": {""name"": ""ge...","[{""from"": ""system"", ""value"": ""You are a deep t...","[""get_future_events""]","You are a deep thinking AI, you may use extrem...","[Retrieve a list of future Azure events, such ...","[get_future_events, get_languages_for_country,..."
1,0,"[{'from': 'system', 'value': 'You are a deep t...","[{""name"": ""get_future_events"", ""description"": ...",Could you provide me with a list of upcoming A...,Get Future Events,ToolAce,"[{""type"": ""function"", ""function"": {""name"": ""ge...","[{""from"": ""system"", ""value"": ""You are a deep t...","[""get_languages_for_country""]","You are a deep thinking AI, you may use extrem...","[Retrieve a list of future Azure events, such ...","[get_future_events, get_languages_for_country,..."
2,1,"[{'from': 'system', 'value': 'You are a deep t...","[{""name"": ""get_future_events"", ""description"": ...",Could you provide me with a list of upcoming A...,Get Future Events,ToolAce,"[{""type"": ""function"", ""function"": {""name"": ""ge...","[{""from"": ""system"", ""value"": ""You are a deep t...","[""get_future_events""]","You are a deep thinking AI, you may use extrem...","[Retrieve a list of future Azure events, such ...","[get_future_events, get_languages_for_country,..."
3,1,"[{'from': 'system', 'value': 'You are a deep t...","[{""name"": ""get_future_events"", ""description"": ...",Could you provide me with a list of upcoming A...,Get Future Events,ToolAce,"[{""type"": ""function"", ""function"": {""name"": ""ge...","[{""from"": ""system"", ""value"": ""You are a deep t...","[""get_languages_for_country""]","You are a deep thinking AI, you may use extrem...","[Retrieve a list of future Azure events, such ...","[get_future_events, get_languages_for_country,..."
4,2,"[{'from': 'system', 'value': 'You are a deep t...","[{""name"": ""get_future_events"", ""description"": ...",Could you provide me with a list of upcoming A...,Get Future Events,ToolAce,"[{""type"": ""function"", ""function"": {""name"": ""ge...","[{""from"": ""system"", ""value"": ""You are a deep t...","[""get_future_events""]","You are a deep thinking AI, you may use extrem...","[Retrieve a list of future Azure events, such ...","[get_future_events, get_languages_for_country,..."
...,...,...,...,...,...,...,...,...,...,...,...,...
32660,14576,"[{'from': 'system', 'value': ""You are a deep t...","[{""type"": ""function"", ""function"": {""name"": ""se...",I'm in New York and I'm craving for some Itali...,Search Restaurants,Glaive,"[{""type"": ""function"", ""function"": {""name"": ""se...","[{""from"": ""system"", ""value"": ""You are a deep t...","[""search_restaurants""]","You are a deep thinking AI, you may use extrem...",[Search for restaurants based on location and ...,"[search_restaurants, calculate_distance]"
32661,14577,"[{'from': 'system', 'value': ""You are a deep t...","[{""type"": ""function"", ""function"": {""name"": ""se...",I'm in New York and I'm craving for some Itali...,Search Restaurants,Glaive,"[{""type"": ""function"", ""function"": {""name"": ""se...","[{""from"": ""system"", ""value"": ""You are a deep t...","[""search_restaurants""]","You are a deep thinking AI, you may use extrem...",[Search for restaurants based on location and ...,"[search_restaurants, calculate_distance]"
32662,14577,"[{'from': 'system', 'value': ""You are a deep t...","[{""type"": ""function"", ""function"": {""name"": ""se...",I'm in New York and I'm craving for some Itali...,Search Restaurants,Glaive,"[{""type"": ""function"", ""function"": {""name"": ""se...","[{""from"": ""system"", ""value"": ""You are a deep t...","[""search_

In [242]:
import numpy as np

In [328]:
class ToolSelectionDataset(Dataset):
    ENTAILMENT_LABEL = 2
    CONTRADICTION_LABEL = 0

    def __init__(self, df: pd.DataFrame, tokenizer):
        self.df = df
        self.tokenizer = tokenizer
        self.X = df['conversation_so_far_str'].tolist()
        self.y = df['tool_names'].tolist()
        self.selected_tools = df['selected_tools'].tolist()
        self.tool_descriptions = df['tool_descriptions'].tolist()

        # Flatten so each training example is a single (premise, tool_description)
        # pair. Rows have a variable number of candidate tools (1-8), so keeping
        # them grouped per-row produced variable-length samples that
        # DataLoader's default collate can't batch (RuntimeError: each element
        # in list of batch should be of equal size).
        self.index = [
            (row, tool_idx)
            for row in range(len(self.df))
            for tool_idx in range(len(self.tool_descriptions[row]))
        ]

    def __len__(self):
        return len(self.index)

    def __getitem__(self, idx):
        row, tool_idx = self.index[idx]
        tool_name = self.y[row][tool_idx]
        label = (
            self.ENTAILMENT_LABEL
            if tool_name in self.selected_tools[row]
            else self.CONTRADICTION_LABEL
        )

        enc = self.tokenizer(
            self.X[row],
            self.tool_descriptions[row][tool_idx],
            return_tensors="pt",
            truncation=True,
            padding="max_length",
            max_length=512,
        )
        # Squeeze the tokenizer's batch dim so every sample is [seq_len],
        # letting default_collate stack them into [batch, seq_len].
        enc = {k: v.squeeze(0) for k, v in enc.items()}
        return enc, torch.tensor(label, dtype=torch.long)

In [329]:
train = data.sample(frac=0.7, random_state=42)
test = data.drop(train.index).sample(frac=0.5, random_state=42)
valid = data.drop(train.index).drop(test.index)

In [330]:
valid

,Unnamed: 0,conversations,tools,task,category,source,tool_list,conversation_so_far,selected_tools,conversation_so_far_str,tool_descriptions,tool_names
9,4,"[{'from': 'system', 'value': 'You are a deep t...","[{""name"": ""get_future_events"", ""description"": ...",Could you provide me with a list of upcoming A...,Get Future Events,ToolAce,"[{""type"": ""function"", ""function"": {""name"": ""ge...","[{""from"": ""system"", ""value"": ""You are a deep t...","[""get_languages_for_country""]","You are a deep thinking AI, you may use extrem...","[Retrieve a list of future Azure events, such ...","[get_future_events, get_languages_for_country,..."
11,5,"[{'from': 'system', 'value': 'You are a deep t...","[{""name"": ""get_future_events"", ""description"": ...",Could you provide me with a list of upcoming A...,Get Future Events,ToolAce,"[{""type"": ""function"", ""function"": {""name"": ""ge...","[{""from"": ""system"", ""value"": ""You are a deep t...","[""get_languages_for_country""]","You are a deep thinking AI, you may use extrem...","[Retrieve a list of future Azure events, such ...","[get_future_events, get_languages_for_country,..."
16,8,"[{'from': 'system', 'value': 'You are a deep t...","[{""name"": ""title_seasons"", ""description"": ""Ret...",Can you tell me the list of currently trending...,Get Trending Tv Shows,ToolAce,"[{""type"": ""function"", ""function"": {""name"": ""ti...","[{""from"": ""system"", ""value"": ""You are a deep t...","[""get_trending_tv_shows""]","You are a deep thinking AI, you may use extrem...",[Retrieve information about TV seasons from Ne...,"[title_seasons, get_genres, get_trending_tv_sh..."
18,9,"[{'from': 'system', 'value': 'You are a deep t...","[{""name"": ""get_future_events"", ""description"": ...",Could you provide me with a list of upcoming A...,Get Future Events,ToolAce,"[{""type"": ""function"", ""function"": {""name"": ""ge...","[{""from"": ""system"", ""value"": ""You are a deep t...","[""get_future_events""]","You are a deep thinking AI, you may use extrem...","[Retrieve a list of future Azure events, such ...","[get_future_events, get_languages_for_country,..."
24,12,"[{'from': 'system', 'value': 'You are a deep t...","[{""name"": ""get_future_events"", ""description"": ...",Could you provide me with a list of upcoming A...,Get Future Events,ToolAce,"[{""type"": ""function"", ""function"": {""name"": ""ge...","[{""from"": ""system"", ""value"": ""You are a deep t...","[""get_future_events""]","You are a deep thinking AI, you may use extrem...","[Retrieve a list of future Azure events, such ...","[get_future_events, get_languages_for_country,..."
...,...,...,...,...,...,...,...,...,...,...,...,...
32629,14561,"[{'from': 'system', 'value': ""You are a deep t...","[{""type"": ""function"", ""function"": {""name"": ""ge...",I need a random number between 1 and 100.,Generate Random Number,Glaive,"[{""type"": ""function"", ""function"": {""name"": ""ge...","[{""from"": ""system"", ""value"": ""You are a deep t...","[""generate_random_number""]","You are a deep thinking AI, you may use extrem...",[Generate a random number within a specified r...,"[generate_random_number, send_email]"
32634,14563,"[{'from': 'system', 'value': ""You are a deep t...","[{""type"": ""function"", ""function"": {""name"": ""ge...",I need a random number between 1 and 100.,Generate Random Number,Glaive,"[{""type"": ""function"", ""function"": {""name"": ""ge...","[{""from"": ""system"", ""value"": ""You are a deep t...","[""generate_random_number""]","You are a deep thinking AI, you may use extrem...",[Generate a random number within a specified r...,"[generate_random_number, send_email]"
32635,14564,"[{'from': 'system', 'value': ""You are a deep t...","[{""type"": ""function"", ""function"": {""name"": ""co...","Hi, I need to convert 1000 US dollars to Euros...",Convert Currency,Glaive,"[{""type"": ""function"", ""function"": {""name"": ""co...","[{""from"": ""system"", ""value"": ""You are a deep t...","[""con

In [331]:
test

,Unnamed: 0,conversations,tools,task,category,source,tool_list,conversation_so_far,selected_tools,conversation_so_far_str,tool_descriptions,tool_names
1802,793,"[{'from': 'system', 'value': 'You are a deep t...","[{""name"": ""extract"", ""description"": ""Extracts ...",I would like to extract details from a LinkedI...,Extract,ToolAce,"[{""type"": ""function"", ""function"": {""name"": ""ex...","[{""from"": ""system"", ""value"": ""You are a deep t...","[""extract""]","You are a deep thinking AI, you may use extrem...",[Extracts data from a LinkedIn URL (prostring ...,"[extract, get_all_motivational_quotes, get_sub..."
2911,1185,"[{'from': 'system', 'value': 'You are a deep t...","[{""name"": ""extract"", ""description"": ""Extracts ...",I would like to extract details from a LinkedI...,Extract,ToolAce,"[{""type"": ""function"", ""function"": {""name"": ""ex...","[{""from"": ""system"", ""value"": ""You are a deep t...","[""extract""]","You are a deep thinking AI, you may use extrem...",[Extracts data from a LinkedIn URL (prostring ...,"[extract, get_all_motivational_quotes, get_sub..."
3862,1522,"[{'from': 'system', 'value': 'You are a deep t...","[{""name"": ""extract"", ""description"": ""Extracts ...",I would like to extract details from a LinkedI...,Extract,ToolAce,"[{""type"": ""function"", ""function"": {""name"": ""ex...","[{""from"": ""system"", ""value"": ""You are a deep t...","[""realtor_school_list""]","You are a deep thinking AI, you may use extrem...",[Extracts data from a LinkedIn URL (prostring ...,"[extract, get_all_motivational_quotes, get_sub..."
7577,2871,"[{'from': 'system', 'value': 'You are a deep t...","[{""name"": ""extract"", ""description"": ""Extracts ...",I would like to extract details from a LinkedI...,Extract,ToolAce,"[{""type"": ""function"", ""function"": {""name"": ""ex...","[{""from"": ""system"", ""value"": ""You are a deep t...","[""realtor_school_list""]","You are a deep thinking AI, you may use extrem...",[Extracts data from a LinkedIn URL (prostring ...,"[extract, get_all_motivational_quotes, get_sub..."
13622,5063,"[{'from': 'system', 'value': 'You are a deep t...","[{""name"": ""extract"", ""description"": ""Extracts ...",I would like to extract details from a LinkedI...,Extract,ToolAce,"[{""type"": ""function"", ""function"": {""name"": ""ex...","[{""from"": ""system"", ""value"": ""You are a deep t...","[""realtor_school_list""]","You are a deep thinking AI, you may use extrem...",[Extracts data from a LinkedIn URL (prostring ...,"[extract, get_all_motivational_quotes, get_sub..."
...,...,...,...,...,...,...,...,...,...,...,...,...
661,330,"[{'from': 'system', 'value': 'You are a deep t...","[{""name"": ""title_seasons"", ""description"": ""Ret...",Can you tell me the list of currently trending...,Get Trending Tv Shows,ToolAce,"[{""type"": ""function"", ""function"": {""name"": ""ti...","[{""from"": ""system"", ""value"": ""You are a deep t...","[""get_genres""]","You are a deep thinking AI, you may use extrem...",[Retrieve information about TV seasons from Ne...,"[title_seasons, get_genres, get_trending_tv_sh..."
362,181,"[{'from': 'system', 'value': 'You are a deep t...","[{""name"": ""get_future_events"", ""description"": ...",Could you provide me with a list of upcoming A...,Get Future Events,ToolAce,"[{""type"": ""function"", ""function"": {""name"": ""ge...","[{""from"": ""system"", ""value"": ""You are a deep t...","[""get_future_events""]","You are a deep thinking AI, you may use extrem...","[Retrieve a list of future Azure events, such ...","[get_future_events, get_languages_for_country,..."
15037,5766,"[{'from': 'system', 'value': 'You are a deep t...","[{""type"": ""function"", ""function"": {""name"": ""ti...",Fetch three cat facts in French.,Defaultroot,Nous-Hermes,"[{""type"": ""function"", ""function"": {""name"": ""ti...","[{""from"": ""system"", ""value"": ""You are a deep t...","[""defaultroot""]","You are a deep thinking AI, you may use extrem...",[\n Returns all

In [332]:
train

,Unnamed: 0,conversations,tools,task,category,source,tool_list,conversation_so_far,selected_tools,conversation_so_far_str,tool_descriptions,tool_names
19244,7869,"[{'from': 'system', 'value': 'You are a deep t...","[{""type"": ""function"", ""function"": {""name"": ""co...",Convert 1.5 cups of flour to grams.,Convert Cooking Measurements,Nous-Hermes,"[{""type"": ""function"", ""function"": {""name"": ""co...","[{""from"": ""system"", ""value"": ""You are a deep t...","[""convert_cooking_measurements""]","You are a deep thinking AI, you may use extrem...",[Converts a quantity of a cooking ingredient f...,"[convert_cooking_measurements, calculate_angle..."
13906,5200,"[{'from': 'system', 'value': 'You are a deep t...","[{""type"": ""function"", ""function"": {""name"": ""ge...",Could you provide the Euro Millions draw detai...,Get Birthday Draws,Nous-Hermes,"[{""type"": ""function"", ""function"": {""name"": ""ge...","[{""from"": ""system"", ""value"": ""You are a deep t...","[""get_birthday_draws""]","You are a deep thinking AI, you may use extrem...",[Fetches lottery draw results for a given birt...,[get_birthday_draws]
27918,12206,"[{'from': 'system', 'value': ""You are a deep t...","[{""type"": ""function"", ""function"": {""name"": ""se...",I have a lot of tomatoes at home. Can you sugg...,Search Recipes,Glaive,"[{""type"": ""function"", ""function"": {""name"": ""se...","[{""from"": ""system"", ""value"": ""You are a deep t...","[""search_recipes""]","You are a deep thinking AI, you may use extrem...",[Search for recipes based on a given ingredien...,"[search_recipes, generate_qr_code]"
31488,13991,"[{'from': 'system', 'value': ""You are a deep t...","[{""type"": ""function"", ""function"": {""name"": ""ge...","Hey, I want to know the latest headlines in sp...",Get News Headlines,Glaive,"[{""type"": ""function"", ""function"": {""name"": ""ge...","[{""from"": ""system"", ""value"": ""You are a deep t...","[""get_news_headlines""]","You are a deep thinking AI, you may use extrem...","[Get the latest news headlines, Calculate the ...","[get_news_headlines, calculate_loan_payment]"
4237,1660,"[{'from': 'system', 'value': 'You are a deep t...","[{""name"": ""extract"", ""description"": ""Extracts ...",I would like to extract details from a LinkedI...,Tool Use,ToolAce,"[{""type"": ""function"", ""function"": {""name"": ""ex...","[{""from"": ""system"", ""value"": ""You are a deep t...","[""get_all_motivational_quotes""]","You are a deep thinking AI, you may use extrem...",[Extracts data from a LinkedIn URL (prostring ...,"[extract, get_all_motivational_quotes, get_sub..."
...,...,...,...,...,...,...,...,...,...,...,...,...
11334,4235,"[{'from': 'system', 'value': 'You are a deep t...","[{""name"": ""extract"", ""description"": ""Extracts ...",I would like to extract details from a LinkedI...,Extract,ToolAce,"[{""type"": ""function"", ""function"": {""name"": ""ex...","[{""from"": ""system"", ""value"": ""You are a deep t...","[""get_all_motivational_quotes""]","You are a deep thinking AI, you may use extrem...",[Extracts data from a LinkedIn URL (prostring ...,"[extract, get_all_motivational_quotes, get_sub..."
14621,5558,"[{'from': 'system', 'value': 'You are a deep t...","[{""type"": ""function"", ""function"": {""name"": ""tt...",Increase the number 5 by one and also do the s...,Get Plus One,Nous-Hermes,"[{""type"": ""function"", ""function"": {""name"": ""tt...","[{""from"": ""system"", ""value"": ""You are a deep t...","[""get_plus_one"", ""get_plus_one""]","You are a deep thinking AI, you may use extrem...",[Converts text to speech using RapidAPI.\n\n ...,"[tts, get_plus_one]"
15332,5913,"[{'from': 'system', 'value': 'You are a deep t...","[{""type"": ""function"", ""function"": {""name"": ""so...",Log in a user named 'user1' with password 'pas...,Loginuser,Nous-Hermes,"[{""type"": ""function"", ""function"": {""name"": ""so...","[{""from"": ""system"", ""value"": ""You are a deep t...","[""software_assets""]","You are a 

In [333]:
training_data = ToolSelectionDataset(train, tok)
testing_data = ToolSelectionDataset(test, tok)
validation_data = ToolSelectionDataset(valid, tok)

In [334]:
train_dataloader = DataLoader(training_data, batch_size=64, shuffle=True)
test_dataloader = DataLoader(testing_data, batch_size=64, shuffle=True)
valid_dataloader = DataLoader(validation_data, batch_size=64, shuffle=True)

In [335]:
# _logits = [ model(input_ids=i["input_ids"], attention_mask=i["attention_mask"]) for i in enc ]

In [336]:
model.train()

BartForSequenceClassification(
  (model): BartModel(
    (shared): Embedding(50265, 1024, padding_idx=1)
    (encoder): BartEncoder(
      (embed_tokens): Embedding(50265, 1024, padding_idx=1)
      (embed_positions): BartLearnedPositionalEmbedding(1026, 1024)
      (layernorm_embedding): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      (layers): ModuleList(
        (0-11): 12 x BartEncoderLayer(
          (self_attn): BartAttention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear(in_features=4096, out_features=1024, bias=T

In [337]:
loss = torch.nn.CrossEntropyLoss(reduction='mean')

In [338]:
optimizer = torch.optim.Adam(model.parameters(), lr=6e-5, weight_decay=0.1)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50, eta_min=6e-6)

In [339]:
print(scheduler.get_last_lr()[0])
curr_loss = None

6e-05


In [1]:
for epoch in range(100):
    print(f"Epoch {epoch+1} training start...")
    model.train()
    epoch_loss = 0.0
    for input_batch, target_batch in tqdm(train_dataloader, desc=f"Epoch {epoch+1}"):
        optimizer.zero_grad()
        logits = model(
            input_ids=input_batch["input_ids"],
            attention_mask=input_batch["attention_mask"],
        )
        batch_loss = loss(logits, target_batch)
        batch_loss.backward()
        optimizer.step()
        epoch_loss += batch_loss.item()

    scheduler.step()
    curr_loss = epoch_loss / len(train_dataloader)
    print(f"Epoch {epoch+1} loss: {curr_loss:.4f}")

Epoch 1 training start...


NameError: name 'model' is not defined

In [325]:
for i in range(10):
    encodings, targets = training_data[i]

    print(len(encodings))
    print(targets)
    # print(
    #     i,
    #     encodings["input_ids"].shape,
    #     targets.shape
    # )

3
[1, 0, 0]
1
[1]
2
[1, 0]
2
[1, 0]
5
[0, 1, 0, 0, 0]
5
[1, 0, 0, 0, 0]
5
[1, 0, 0, 0, 0]
2
[1, 0]
5
[0, 1, 0, 0, 0]
2
[1, 0]


In [326]:
data['tool_descriptions'][0]

['Retrieve a list of future Azure events, such as maintenance windows, upstrings, or other scheduled events.',
 'Get a list of valid languages for a given country code.',
 'This endpoint returns a list of all available dog breeds, along with their relevant information.',
 "Retrieves a LinkedIn user's prostring data, including experience, education history, skills, and company-related details, given their Sales Navigator URL.",
 'Retrieve a list of the top 100 companies that are related to a given SIC code.',
 "Returns every verse containing the supplied Strong's number. Include LXX boolean option allows searching the Septuagint translation of the Old Testament when searching for a Greek word, enabling connections between New Testament words and Old Testament concepts."]